In [ ]:
# Purpose: imports and small setup
import logging
logger = logging.getLogger(__name__)
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')


In [ ]:
# Purpose: plotting / figure generation
# Reproducibility helper added by tools/add_repro_cell.py
import os, sys, random, json
from datetime import datetime
import numpy as np
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
# Optional: set torch deterministic flags if torch is available
try:
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # deterministic settings (may affect performance)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
except Exception:
    pass
# Record environment info for reproducibility
info = dict(
    python=sys.executable,
    python_version='.'.join(map(str, sys.version_info[:3])),
    numpy=np.__version__,
    seed=seed,
    timestamp=str(datetime.utcnow())
)
# Add optional package checks
try:
    import pkg_resources
    pkgs=['nbformat','pandas','scipy','sklearn','matplotlib','seaborn']
    pv = {p: pkg_resources.get_distribution(p).version for p in pkgs if pkg_resources.get_distribution(p) is not None}
    info['packages'] = pv
except Exception:
    pass
logger.info('REPRO: ' + json.dumps(info))


In [1]:
# Purpose: imports and small setup
from google.colab import drive
from pathlib import Path
import sys

# (commented) drive.mount("/content/drive")

# (sanitized path) PROJECT_ROOT = Path("PROJECT_ROOT")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root missing: {PROJECT_ROOT}")

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"sys.path entries added: {PROJECT_ROOT}, {PROJECT_ROOT / 'scripts'}")


Mounted at /content/drive
Project root: /content/drive/MyDrive/Dr Dacon ICDM/SeamSFF-main
sys.path entries added: /content/drive/MyDrive/Dr Dacon ICDM/SeamSFF-main, /content/drive/MyDrive/Dr Dacon ICDM/SeamSFF-main/scripts


In [2]:
# Purpose: data processing / analysis step
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
expected_scripts = [
    "download_benchmark_candidates.py",
    "build_python_task_pool.py",
    "build_clinical_task_pool.py",
    "build_history_task_pool.py",
    "pipeline_utils.py",
    "python_execution_audit.py",
]

for name in expected_scripts:
    path = SCRIPTS_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Script missing: {path}")
    size_kb = path.stat().st_size / 1024
    logger.info(f"{name:40s}  {size_kb:6.1f} KB")


download_benchmark_candidates.py            12.0 KB
build_python_task_pool.py                   10.4 KB
build_clinical_task_pool.py                 15.6 KB
build_history_task_pool.py                  22.1 KB
pipeline_utils.py                            8.1 KB
python_execution_audit.py                    4.6 KB


In [3]:
# Purpose: imports and small setup
import subprocess
import json

script_path = SCRIPTS_DIR / "download_benchmark_candidates.py"

result = subprocess.run(
    ["python", str(script_path), "--registry-only"],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
)

# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')
    logger.info("STDOUT:")
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')
    logger.info("STDERR:")
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')

registry_path = PROJECT_ROOT / "configs" / "benchmark_source_registry.json"
with registry_path.open() as f:
    registry = json.load(f)

logger.info(f"\nRegistry has {len(registry)} benchmark sources:")
for entry in registry:
    logger.info(f"  {entry['benchmark_source']:30s}  domain={entry['domain']:30s}  method={entry['method']}")


Exit code: 0
STDOUT:
{
  "registry": "configs/benchmark_source_registry.json",
  "status": "written"
}


Registry has 10 benchmark sources:
  MBPP                            domain=python_debugging                method=direct_files
  HumanEval                       domain=python_debugging                method=direct_files
  APPS                            domain=python_debugging                method=direct_files
  MedQA                           domain=clinical_text_interpretation    method=hf_rows_multi_split
  MedMCQA                         domain=clinical_text_interpretation    method=hf_rows
  PubMedQA                        domain=clinical_text_interpretation    method=direct_files
  MMLU medical subset             domain=clinical_text_interpretation    method=hf_rows_multi_config
  World-History-1500 QA           domain=historical_document_synthesis   method=direct_files
  ArchivalQA                      domain=historical_document_synthesis   method=hf_rows
  ChroniclingAmeri

In [4]:
# Purpose: imports and small setup
import subprocess
import time

script_path = SCRIPTS_DIR / "download_benchmark_candidates.py"

start = time.time()
result = subprocess.run(
    ["python", str(script_path)],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
)
elapsed = time.time() - start

# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')
logger.info(f"Elapsed: {elapsed:.1f}s")
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')
    logger.info("STDERR:")
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')

summary_path = PROJECT_ROOT / "configs" / "benchmark_download_summary.json"
if summary_path.exists():
    with summary_path.open() as f:
        summary = json.load(f)
    logger.info(f"\nDownload summary ({len(summary)} sources):")
    for entry in summary:
        status = entry.get("status") or "downloaded"
        outputs = entry.get("outputs", [])
        if outputs:
            for out in outputs:
                row_info = f" rows={out.get('rows')}" if "rows" in out else ""
                logger.info(f"  {entry['benchmark_source']:25s}  {out['status']}  {out['path']}{row_info}")
        else:
            logger.info(f"  {entry['benchmark_source']:25s}  {status}")


Exit code: 0
Elapsed: 14.9s

Download summary (10 sources):
  MBPP                       downloaded  python_domain/raw/mbpp/sanitized-mbpp.json
  HumanEval                  downloaded  python_domain/raw/humaneval/HumanEval.jsonl
  APPS                       skipped_large
  MedQA                      downloaded  clinical_domain/raw/medqa/train.jsonl rows=600
  MedQA                      downloaded  clinical_domain/raw/medqa/test.jsonl rows=600
  MedMCQA                    downloaded  clinical_domain/raw/medmcqa/train.jsonl rows=600
  PubMedQA                   downloaded  clinical_domain/raw/pubmedqa/ori_pqal.json
  MMLU medical subset        downloaded  clinical_domain/raw/mmlu/clinical_knowledge_test.jsonl rows=265
  MMLU medical subset        downloaded  clinical_domain/raw/mmlu/college_medicine_test.jsonl rows=173
  MMLU medical subset        downloaded  clinical_domain/raw/mmlu/medical_genetics_test.jsonl rows=100
  MMLU medical subset        downloaded  clinical_domain/raw/mmlu/pr

In [5]:
# Purpose: imports and small setup
import subprocess
import time

builders = [
    "build_python_task_pool.py",
    "build_clinical_task_pool.py",
    "build_history_task_pool.py",
]

for builder in builders:
    logger.info(f"\n{'=' * 60}")
    logger.info(f"Running {builder}")
    logger.info('=' * 60)
    start = time.time()
    result = subprocess.run(
        ["python", str(SCRIPTS_DIR / builder)],
        capture_output=True,
        text=True,
        cwd=str(PROJECT_ROOT),
    )
    elapsed = time.time() - start
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')
        logger.info("STDERR:")
# (condensed debug output inserted by clean_notebooks)
logger.info(f'Command exit={result.returncode}, stdout_len={len(result.stdout) if result.stdout else 0}, stderr_len={len(result.stderr) if result.stderr else 0}')



Running build_python_task_pool.py
Exit code: 0  (0.9s)
{
  "mbpp_selected_60.jsonl": {
    "count": 60,
    "status": "written"
  },
  "humaneval_selected_60.jsonl": {
    "count": 60,
    "status": "written"
  },
  "apps_selected_60.jsonl": {
    "count": 0,
    "status": "raw files not found"
  }
}


Running build_clinical_task_pool.py
Exit code: 0  (0.9s)
{
  "medmcqa_selected_60.jsonl": {
    "count": 60,
    "status": "written"
  },
  "mmlu_medical_selected_60.jsonl": {
    "count": 60,
    "status": "written"
  },
  "pubmedqa_selected_60.jsonl": {
    "count": 60,
    "status": "written"
  },
  "medqa_selected_60.jsonl": {
    "count": 60,
    "status": "written"
  }
}


Running build_history_task_pool.py
Exit code: 0  (0.6s)
{
  "World-History-1500 QA": {
    "count": 60,
    "difficulty_counts": {
      "easy": 20,
      "medium": 20,
      "hard": 20
    }
  },
  "ArchivalQA": {
    "count": 60,
    "difficulty_counts": {
      "easy": 20,
      "medium": 20,
      "hard": 20

In [6]:
# Purpose: imports and small setup
import json
from pathlib import Path

SELECTED_FILES = {
    "python": PROJECT_ROOT / "python_domain" / "selected",
    "clinical": PROJECT_ROOT / "clinical_domain" / "selected",
    "history": PROJECT_ROOT / "history_domain" / "selected",
}

for domain, selected_dir in SELECTED_FILES.items():
    if not selected_dir.exists():
        logger.info(f"{domain}: directory missing — {selected_dir}")
        continue
    logger.info(f"\n{'=' * 60}")
    logger.info(f"Domain: {domain}")
    logger.info('=' * 60)
    for jsonl_path in sorted(selected_dir.glob("*.jsonl")):
        with jsonl_path.open() as f:
            records = [json.loads(line) for line in f if line.strip()]
        logger.info(f"\n{jsonl_path.name}: {len(records)} records")
        if not records:
            continue
        sample = records[0]
        logger.info(f"  fields: {sorted(sample.keys())}")
        for tier in ("easy", "medium", "hard"):
            count = sum(1 for r in records if r.get("difficulty_tier") == tier)
            logger.info(f"  {tier}: {count}")



Domain: python

humaneval_selected_60.jsonl: 60 records
  fields: ['adaptation_note', 'answer', 'benchmark_source', 'benchmark_split', 'contamination_note', 'context', 'difficulty_assignment_method', 'difficulty_review_stage', 'difficulty_tier', 'domain', 'ground_truth', 'license_or_access_note', 'question', 'rubric', 'selected_for_pilot', 'source_url_or_citation', 'task_id', 'task_prompt', 'task_type', 'tests', 'topic']
  easy: 20
  medium: 20
  hard: 20

mbpp_selected_60.jsonl: 60 records
  fields: ['adaptation_note', 'answer', 'benchmark_source', 'benchmark_split', 'contamination_note', 'context', 'difficulty_assignment_method', 'difficulty_review_stage', 'difficulty_tier', 'domain', 'ground_truth', 'license_or_access_note', 'question', 'rubric', 'selected_for_pilot', 'source_url_or_citation', 'task_id', 'task_prompt', 'task_type', 'tests', 'topic']
  easy: 20
  medium: 20
  hard: 20

python_domain_selected.jsonl: 120 records
  fields: ['adaptation_note', 'answer', 'benchmark_sourc

In [7]:
# Purpose: imports and small setup
import json

SAMPLE_FILES = {
    "python":   PROJECT_ROOT / "python_domain"   / "selected" / "mbpp_selected_60.jsonl",
    "clinical": PROJECT_ROOT / "clinical_domain" / "selected" / "medqa_selected_60.jsonl",
    "history":  PROJECT_ROOT / "history_domain"  / "selected" / "archivalqa_selected_60.jsonl",
}

for domain, path in SAMPLE_FILES.items():
    with path.open() as f:
        first = json.loads(f.readline())
    logger.info(f"\n{'=' * 60}")
    logger.info(f"{domain}  ({path.name})")
    logger.info('=' * 60)
    logger.info(f"task_id:           {first['task_id']}")
    logger.info(f"benchmark_source:  {first['benchmark_source']}")
    logger.info(f"difficulty_tier:   {first['difficulty_tier']}")
    logger.info(f"task_type:         {first['task_type']}")
    logger.info()
    logger.info(f"task_prompt:")
    logger.info(first['task_prompt'][:400] + ("..." if len(first['task_prompt']) > 400 else ""))
    logger.info()
    logger.info(f"ground_truth:")
    gt = first['ground_truth'] if isinstance(first['ground_truth'], str) else json.dumps(first['ground_truth'], indent=2)
    logger.info(gt[:400] + ("..." if len(gt) > 400 else ""))



python  (mbpp_selected_60.jsonl)
task_id:           python_mbpp_sanitized-mbpp_127
benchmark_source:  MBPP
difficulty_tier:   easy
task_type:         code_generation

task_prompt:
Solve the following Python benchmark task. Provide correct Python code and briefly explain the reasoning.

Task: Write a function to multiply two integers.

ground_truth:
{
  "reference_solution": "def multiply_int(x, y): if y < 0: return -multiply_int(x, -y) elif y == 0: return 0 elif y == 1: return x else: return x + multiply_int(x, y - 1)",
  "tests": [
    "assert multiply_int(10,20)==200",
    "assert multiply_int(5,10)==50",
    "assert multiply_int(4,8)==32"
  ]
}

clinical  (medqa_selected_60.jsonl)
task_id:           clinical_medqa_train_000577
benchmark_source:  MedQA
difficulty_tier:   easy
task_type:         medical_qa

task_prompt:
Answer the MedQA benchmark question and choose the best option.

Question: In which of the following pathological states would the oxygen content of the trachea resem

In [ ]:
# Purpose: empty cell
